# Тестовое задание

### Предлагаемый подход

В рамках предложенного задания по ранжированию существует 3 типа подходов: pointwise, pairwise и listwise. Я выбрала pairwise подход, так как он эффективнее pointwise и стабильнее listwise. Использую модель CatBoost (может работать с текстовыми признаками) с функцией потерь YetiRankPairwise.

### Настройка окружения

Подключим все нужные библиотеки и зафиксируем seed для детерминированности.

In [1]:
import pandas as pd
import numpy as np
import random
import os
import gc
from catboost import CatBoostRanker, Pool
from sklearn.model_selection import GroupShuffleSplit

SEED = 67
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)

### Загрузка данных

Загрузим данные для обучения и тестирования с Google Drive.

In [2]:
!pip install -q gdown

In [3]:
!gdown --folder "https://drive.google.com/drive/folders/17zhM4Cq6sWL10tFIZO7XFsIhgNIZICtj"

Retrieving folder contents
Processing file 1LAVAOCeyPI3feJ_IhrR3x4U-lAdO-OiE catboost_model (1).cbm
Processing file 1xS1-h3tfzFxhzOeJZP-uaCGTJSKYFv20 test-dset-small.parquet
Processing file 1Bwtz_wxNaR70ZZrcQPoJHETWKislsL__ train-dset.parquet
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1LAVAOCeyPI3feJ_IhrR3x4U-lAdO-OiE
To: /kaggle/working/avito-ds/catboost_model (1).cbm
100%|██████████████████████████████████████| 13.7M/13.7M [00:00<00:00, 56.6MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1xS1-h3tfzFxhzOeJZP-uaCGTJSKYFv20
From (redirected): https://drive.google.com/uc?id=1xS1-h3tfzFxhzOeJZP-uaCGTJSKYFv20&confirm=t&uuid=91eb0efd-7955-49b3-a487-b162c0520060
To: /kaggle/working/avito-ds/test-dset-small.parquet
100%|████████████████████████████████████████| 152M/152M [00:01<00:00, 90.4MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1Bw

In [4]:
import pandas as pd

train = pd.read_parquet('/kaggle/working/avito-ds/train-dset.parquet').drop(columns=['item_description'])
test = pd.read_parquet('/kaggle/working/avito-ds/test-dset-small.parquet').drop(columns=['item_description'])

Проверим на наличие пропусков в данных:

In [5]:
train.isna().sum()

query_id                       0
item_id                        0
query_text                     0
item_title                   107
query_cat                      0
query_mcat               1761233
query_loc                      0
item_cat_id                    0
item_mcat_id                   0
item_loc                       0
price                          0
item_query_click_conv          0
item_contact                   0
dtype: int64

In [6]:
test.isna().sum()

query_id                     0
item_id                      0
query_text                   0
item_title                   2
query_cat                    0
query_mcat               76025
query_loc                    0
item_cat_id                  0
item_mcat_id                 0
item_loc                     0
price                        0
item_query_click_conv        0
dtype: int64

### Генерация признаков

Создадим дополнительные признаки для улучшения работы модели. Как известно, наиболее информативными признаками являются признаки, описывающие связь между запросом и документом, поэтому добавим флаги совпадения категорий и локаций (`cat_match`, `mcat_match`, `loc_match`), цену товара относительно средней в этом запросе, а также ее ранг. Обработаем пропуски. Также сделаем downcast типов для уменьшения потребления памяти (иначе превысим лимиты в kaggle :c).

In [7]:
def reduce_mem_usage(df):
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col] = df[col].astype('float32')
        if df[col].dtype == 'int64':
            df[col] = df[col].astype('int32')
    return df

def create_features(df):
    df['item_title'] = df['item_title'].fillna('')
    df['query_mcat'] = df['query_mcat'].fillna(-1).astype(np.int32)
    
    group_price_mean = df.groupby('query_id')['price'].transform('mean')
    df['rel_price'] = df['price'] / (group_price_mean + 1e-6)
    df['price_rank'] = df.groupby('query_id')['price'].rank(method='dense')
    
    df['cat_match'] = (df['query_cat'] == df['item_cat_id']).astype(np.int8)
    df['mcat_match'] = (df['query_mcat'] == df['item_mcat_id']).astype(np.int8)
    df['loc_match'] = (df['query_loc'] == df['item_loc']).astype(np.int8)
    
    df['query_len'] = df['query_text'].str.len().astype(np.int16)
    df['title_len'] = df['item_title'].str.len().astype(np.int16)
    
    return reduce_mem_usage(df)

### Препроцессинг

Применим функцию генерации признаков и подготовим данные для обучения. Отсортируем строки по запросам для использования CatBoost.

In [8]:
train = create_features(train)
test = create_features(test)

feature_cols = [
    'price', 'item_query_click_conv', 'rel_price', 'price_rank',
    'cat_match', 'mcat_match', 'loc_match', 'query_len', 'title_len'
]
text_features = ['query_text', 'item_title']

train = train.sort_values(by='query_id').reset_index(drop=True)
test = test.sort_values(by='query_id').reset_index(drop=True)

Разделим данные на трейн и валидацию:

In [9]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=SEED)
train_idx, val_idx = next(gss.split(train, groups=train['query_id']))

train_data = train.iloc[train_idx]
val_data = train.iloc[val_idx]

In [10]:
train_pool = Pool(
    data=train_data[feature_cols + text_features],
    label=train_data['item_contact'],
    group_id=train_data['query_id'],
    text_features=text_features
)

val_pool = Pool(
    data=val_data[feature_cols + text_features],
    label=val_data['item_contact'],
    group_id=val_data['query_id'],
    text_features=text_features
)

Очистим память:

In [11]:
import gc
del train, train_data, val_data
gc.collect()

0

### Обучение модели

In [13]:
model = CatBoostRanker(
    iterations=1500,
    learning_rate=0.08,
    depth=7,
    loss_function='YetiRankPairwise',
    task_type='GPU',
    random_seed=SEED,
    verbose=100,
    early_stopping_rounds=100
)

model.fit(
    train_pool,
    eval_set=val_pool
)

Default metric period is 5 because PFound is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	test: 0.1834159	best: 0.1834159 (0)	total: 978ms	remaining: 24m 25s
100:	test: 0.2122503	best: 0.2122503 (100)	total: 50.6s	remaining: 11m 41s
200:	test: 0.2136971	best: 0.2136971 (200)	total: 1m 28s	remaining: 9m 34s
300:	test: 0.2142863	best: 0.2143058 (293)	total: 2m 5s	remaining: 8m 18s
400:	test: 0.2146581	best: 0.2146581 (400)	total: 2m 40s	remaining: 7m 21s
500:	test: 0.2150225	best: 0.2150225 (500)	total: 3m 16s	remaining: 6m 30s
600:	test: 0.2152301	best: 0.2152301 (600)	total: 3m 51s	remaining: 5m 45s
700:	test: 0.2153982	best: 0.2154042 (699)	total: 4m 25s	remaining: 5m 2s
800:	test: 0.2156296	best: 0.2156427 (793)	total: 5m	remaining: 4m 21s
900:	test: 0.2159076	best: 0.2159076 (900)	total: 5m 34s	remaining: 3m 42s
1000:	test: 0.2160343	best: 0.2160384 (989)	total: 6m 9s	remaining: 3m 4s
1100:	test: 0.2162428	best: 0.2162878 (1095)	total: 6m 43s	remaining: 2m 26s
1200:	test: 0.2164287	best: 0.2164554 (1198)	total: 7m 18s	remaining: 1m 49s
1300:	test: 0.2165425	best: 0.21

### Тестирование модели

In [ ]:
# если необходимо чисто протестировать модель, можно загрузить веса здесь
'''
model = CatBoostRanker()
model.load_model('/kaggle/working/avito-ds/catboost_model.cbm')
'''

In [17]:
test_pool = Pool(
    data=test[feature_cols + text_features],
    group_id=test['query_id'],
    text_features=text_features
)

test['score'] = model.predict(test_pool)

submission_df = test[['query_id', 'item_id', 'score']].sort_values(
    by=['query_id', 'score'], 
    ascending=[True, False]
)

submission_df[['query_id', 'item_id']].to_csv(
    'solution.csv', 
    header=['query_id', 'item_id'], 
    index=False
)

In [16]:
model.save_model("catboost_model.cbm")

### Улучшение модели

В результате удалось добиться NDCG@10=56.924% для тестового набора данных. Одним из направлений улучшения модели можно отметить использование описания объявления, которое в моем решении не было использовано совсем для экономии памяти. Также можно было провести большее количество экспериментов для более точного определения гиперпараметров обучения, однако ограниченное количество посылок не позволило это сделать.